# Plot scGPT

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import pickle
import polars as pl
import pandas as pd
from deeptan.stat.metrics import ClusteringMetricsCalculator, transform_ct_df, read_batch_from_h5ad

/home/wuch/miniforge3/envs/sc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load cell emb

In [2]:
path_scgpt_results = "/mnt/hdd2/homext/wuch/xn2p/run/predict/scgpt/"
true_data_dir = "/mnt/hdd2/homext/wuch/xn2p/data/raw_df/scRNA/GSE226097_Annotated_split_strata"

_dataset = "snrna"
# _dataset = "scmul"

_type = "cell_emb"
# _type = "gene_emb"

_split = "test"
_seed = 42
n_feat = 1600

In [3]:
_path_embs = os.path.join(path_scgpt_results, f"{_dataset}_{_type}")
_files = os.listdir(_path_embs)
_path_emb = [i for i in _files if i.endswith(f"{_split}{_seed}_{n_feat}_{_type}.pkl")][0]
_path_emb = os.path.join(_path_embs, _path_emb)
_emb = pd.read_pickle(_path_emb)
# with open(_path_emb, "rb") as f:
#     _emb = pickle.load(f)
print(type(_emb))
print(_emb.shape)
# print(_emb.head())

<class 'pandas.core.frame.DataFrame'>
(679, 128)


## Load true labels

In [4]:
i=2

_path = os.path.join(true_data_dir, f"split_{_seed}_{i}.parquet")
_obs_names = pl.read_parquet(_path).select(["obs_names"])

_labels_df = pl.read_parquet(os.path.join(true_data_dir, "celltypes_onehot.parquet")).rename({"bc": "obs_names"})
# Pick obs_names that are in the split and keep the order
_labels_df = _obs_names.join(_labels_df, on="obs_names", how="left")
_labels_ct = transform_ct_df(_labels_df)
# print(_labels_df)
# print(_labels_ct)

## Load batch info

In [5]:
path_h5ad = os.path.join(true_data_dir, "origin.h5ad")
_batch_df = read_batch_from_h5ad(path_h5ad)
_batch_df = _obs_names.join(_batch_df, on="obs_names", how="left")
print(_batch_df.columns)

['obs_names', 'batch']


## Metrics for clustering

In [10]:
metricscalc = ClusteringMetricsCalculator(
    true_labels=_labels_df.drop("obs_names").to_numpy(),
    pred_labels=None,
    batches=_batch_df["batch"].to_numpy(),
    X_emb=_emb.to_numpy(),
)
metrics_clustering_dict = metricscalc.calculate_metrics()
print(metrics_clustering_dict)
metrics_clustering_df = pl.DataFrame(metrics_clustering_dict)

{'kbet': np.float64(0.9808541973490427), 'asw_true_label': np.float32(0.4954783), 'asw_pred_label': np.float32(0.50876635), 'ari': -0.0012410103784208464, 'nmi': np.float64(0.0068514622282706255), 'ami': np.float64(-0.0010100949821803434), 'ari_leiden': -0.0012410103784208464, 'nmi_leiden': np.float64(0.0068514622282706255), 'ami_leiden': np.float64(-0.0010100949821803434)}
